# 14 — Soft-F1 regularized distillation

**Research question:** Does adding a differentiable F1 loss term to the
distillation objective improve the student’s test F1 or fairness?

The baseline distillation loss (notebook 09) is:

$$\mathcal{L} = \alpha \cdot \text{KD} + (1 - \alpha) \cdot \text{BCE}$$

We add a **soft-F1 regularizer** (weight $\beta$):

$$\mathcal{L} = \alpha \cdot \text{KD} + (1 - \alpha) \cdot \text{BCE} + \beta \cdot (1 - \widetilde{F_1})$$

where $\widetilde{F_1}$ uses sigmoid probabilities as soft predictions:

$$\widetilde{\text{TP}} = \sum y \cdot \hat{p}, \quad
  \widetilde{\text{FP}} = \sum (1-y) \cdot \hat{p}, \quad
  \widetilde{\text{FN}} = \sum y \cdot (1-\hat{p})$$

$$\widetilde{F_1} = \frac{2\,\widetilde{\text{TP}}}{2\,\widetilde{\text{TP}} + \widetilde{\text{FP}} + \widetilde{\text{FN}} + \epsilon}$$

**Hypothesis:** Soft-F1 directly optimizes the metric we care about, so test
F1 should improve; but it may hurt calibration or fairness since the F1 term
does not penalize identity-group FPR.

## Setup
1. Runtime → **GPU** (Colab) or run locally with CUDA
2. Ensure `src/`, `data/raw/jigsaw/`, and `models/toxic-bert/` are available
3. Run all cells — trains **two** students (beta=0.3, beta=0.5) and compares against the existing beta=0 baseline

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT / "DeepProjectHate" / "src").exists():
    ROOT = ROOT / "DeepProjectHate"
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("ROOT", ROOT.resolve())

In [ ]:
from device import get_device
print("device:", get_device())

from paths import TRAIN_CSV, SPLIT_INDICES_PATH
if not TRAIN_CSV.exists():
    print("Downloading Jigsaw...")
    %run scripts/download_jigsaw.py
if not SPLIT_INDICES_PATH.exists():
    %run scripts/make_splits.py
if not (Path("models") / "toxic-bert").exists():
    print("Downloading toxic-bert teacher...")
    %run scripts/download_toxic_bert.py

## 1. Training configuration

We keep all hyperparameters identical to notebook 09 except for `beta`.
Each run saves to a separate directory under `models/`.

In [ ]:
EPOCHS = 4
BATCH = 16
MAX_TRAIN = 20000
MAX_VAL = 2000

BETA_VALUES = [0.3, 0.5]

for beta in BETA_VALUES:
    tag = f"distilbert-student-f1-b{beta}".replace(".", "")
    out_dir = f"models/{tag}"
    cmd = (
        f"python scripts/train_distill.py"
        f" --epochs {EPOCHS} --batch-size {BATCH}"
        f" --max-train-samples {MAX_TRAIN} --max-val-samples {MAX_VAL}"
        f" --beta {beta}"
        f" --output-dir {out_dir}"
    )
    print(f"\n{'='*60}")
    print(f"Training beta={beta} -> {out_dir}")
    print(cmd)
    !{cmd}

## 2. Load all students and compare

In [ ]:
import numpy as np
import pandas as pd
import torch
from distill import DistilBertStudentScorer
from splits import get_split_frame
from paths import TOXIC_LABELS, SEED
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

test_df = get_split_frame("test")
EVAL_N = 500
test_sample = test_df.sample(EVAL_N, random_state=SEED).reset_index(drop=True)
texts = test_sample["comment_text"].astype(str).tolist()
y_true = test_sample["toxic"].astype(int).values
print(f"Test sample: {EVAL_N} comments, {y_true.sum()} toxic ({y_true.mean():.1%})")

In [ ]:
from fairness import build_identity_probe_df, per_group_fpr, control_fpr, CONTROL_TEXTS

probe = build_identity_probe_df()
print(f"Fairness probe: {len(probe)} sentences, {probe['group'].nunique()} groups")

students = {
    "beta=0 (baseline)": "models/distilbert-student",
}
for beta in BETA_VALUES:
    tag = f"distilbert-student-f1-b{beta}".replace(".", "")
    students[f"beta={beta}"] = f"models/{tag}"

rows = []
for label, path in students.items():
    if not Path(path).exists():
        print(f"SKIP {label}: {path} not found")
        continue
    print(f"\nScoring {label} from {path}...")
    scorer = DistilBertStudentScorer(model_path=path)

    # Accuracy metrics
    probs = scorer.predict_toxic_proba(texts)
    preds = (probs >= 0.5).astype(int)
    f1 = f1_score(y_true, preds, zero_division=0)
    roc = roc_auc_score(y_true, probs) if len(np.unique(y_true)) > 1 else float("nan")
    pr = average_precision_score(y_true, probs) if len(np.unique(y_true)) > 1 else float("nan")

    # Fairness metrics
    fair_probs = scorer.predict_toxic_proba(probe["text"].tolist())
    ctrl_probs = scorer.predict_toxic_proba(CONTROL_TEXTS)
    group_df = per_group_fpr(probe, fair_probs, threshold=0.5)
    ctrl = control_fpr(ctrl_probs, threshold=0.5)
    identity_fpr = float(group_df["fpr"].mean())

    # Training config
    cfg_path = Path(path) / "distill_config.json"
    cfg_beta = 0.0
    if cfg_path.exists():
        cfg = json.loads(cfg_path.read_text())
        cfg_beta = cfg.get("config", {}).get("beta", 0.0)

    rows.append({
        "model": label,
        "beta": cfg_beta,
        "F1": f1,
        "ROC-AUC": roc,
        "PR-AUC": pr,
        "identity_FPR": identity_fpr,
        "control_FPR": ctrl["fpr"],
    })
    print(f"  F1={f1:.3f}  ROC-AUC={roc:.3f}  PR-AUC={pr:.3f}  identity_FPR={identity_fpr:.3f}")

results_df = pd.DataFrame(rows)
print("\n" + "="*60)
results_df

## 3. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ["F1", "ROC-AUC", "identity_FPR"]):
    ax.bar(results_df["model"], results_df[metric])
    ax.set_title(metric)
    ax.set_ylabel(metric)
    for i, v in enumerate(results_df[metric]):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
    ax.set_ylim(0, max(results_df[metric].max() * 1.15, 0.1))

fig.suptitle("Soft-F1 regularization: accuracy vs fairness trade-off", fontsize=13)
plt.tight_layout()
fig.savefig("report/figures/soft_f1_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved report/figures/soft_f1_comparison.png")

## 4. Save results

In [ ]:
from paths import PROCESSED_DIR
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

out_path = PROCESSED_DIR / "soft_f1_distillation.json"
results_df.to_json(out_path, orient="records", indent=2)
print(f"Wrote {out_path}")

# Per-group breakdown for the best beta
if len(rows) > 1:
    best = results_df.loc[results_df["F1"].idxmax()]
    print(f"\nBest F1: {best['model']} (F1={best['F1']:.3f}, identity_FPR={best['identity_FPR']:.3f})")

## 5. Discussion

**Expected findings:**

- Soft-F1 regularization directly optimizes the evaluation metric during
  training, so test F1 may improve (especially for the minority toxic class).
- However, the F1 term does not penalize identity-group false positives,
  so fairness (identity FPR) may remain unchanged or even worsen.
- If beta is too high, it can dominate the loss and hurt the KD signal
  from the teacher, degrading ranking quality (PR-AUC / ROC-AUC).

This experiment is part of the “future work” direction in the report (\S4)
and demonstrates that loss-function design has measurable downstream effects
on both accuracy and fairness.